# 05 — Train Gan / 生成对抗网络

**Chapter 09 — File 5 of 11 / 第09章 — 第5个文件（共11个）**

---

## Summary / 总结

This script demonstrates **example of a gan for generating faces**.

本脚本演示 **example of a gan for generating faces**。

---
## Step 1 — example of a gan for generating faces

In [ ]:
from numpy import load
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from matplotlib import pyplot

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(in_shape=(80,80,3)):
	model = Sequential()

---
## Step 3 — normal

In [ ]:
model.add(Conv2D(128, (5,5), padding='same', input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 4 — downsample to 40x40

In [ ]:
model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 5 — downsample to 20x30

In [ ]:
model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 6 — downsample to 10x10

In [ ]:
model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 7 — downsample to 5x5

In [ ]:
model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 8 — classifier

In [ ]:
model.add(Flatten())
	model.add(Dropout(0.4))
	model.add(Dense(1, activation='sigmoid'))

---
## Step 9 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

---
## Step 10 — define the standalone generator model

In [ ]:
def define_generator(latent_dim):
	model = Sequential()

---
## Step 11 — foundation for 5x5 feature maps

In [ ]:
n_nodes = 128 * 5 * 5
	model.add(Dense(n_nodes, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((5, 5, 128)))

---
## Step 12 — upsample to 10x10

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 13 — upsample to 20x20

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 14 — upsample to 40x40

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 15 — upsample to 80x80

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 16 — output layer 80x80x3

In [ ]:
model.add(Conv2D(3, (5,5), activation='tanh', padding='same'))
	return model

---
## Step 17 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model):

---
## Step 18 — make weights in the discriminator not trainable

In [ ]:
d_model.trainable = False

---
## Step 19 — connect them

In [ ]:
model = Sequential()

---
## Step 20 — add generator

In [ ]:
model.add(g_model)

---
## Step 21 — add the discriminator

In [ ]:
model.add(d_model)

---
## Step 22 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

---
## Step 23 — load and prepare training images

In [ ]:
def load_real_samples():

---
## Step 24 — load the face dataset

In [ ]:
data = load('img_align_celeba.npz')
	X = data['arr_0']

---
## Step 25 — convert from unsigned ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 26 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	return X

---
## Step 27 — select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 28 — choose random instances

In [ ]:
ix = randint(0, dataset.shape[0], n_samples)

---
## Step 29 — retrieve selected images

In [ ]:
X = dataset[ix]

---
## Step 30 — generate 'real' class labels (1)

In [ ]:
y = ones((n_samples, 1))
	return X, y

---
## Step 31 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples):

---
## Step 32 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 33 — reshape into a batch of inputs for the network

In [ ]:
x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

---
## Step 34 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(g_model, latent_dim, n_samples):

---
## Step 35 — generate points in latent space

In [ ]:
x_input = generate_latent_points(latent_dim, n_samples)

---
## Step 36 — predict outputs

In [ ]:
X = g_model.predict(x_input)

---
## Step 37 — create 'fake' class labels (0)

In [ ]:
y = zeros((n_samples, 1))
	return X, y

---
## Step 38 — create and save a plot of generated images

In [ ]:
def save_plot(examples, epoch, n=10):

---
## Step 39 — scale from [-1,1] to [0,1]

In [ ]:
examples = (examples + 1) / 2.0

---
## Step 40 — plot images

In [ ]:
for i in range(n * n):

---
## Step 41 — define subplot

In [ ]:
pyplot.subplot(n, n, 1 + i)

---
## Step 42 — turn off axis

In [ ]:
pyplot.axis('off')

---
## Step 43 — plot raw pixel data

In [ ]:
pyplot.imshow(examples[i])

---
## Step 44 — save plot to file

In [ ]:
filename = 'generated_plot_e%03d.png' % (epoch+1)
	pyplot.savefig(filename)
	pyplot.close()

---
## Step 45 — evaluate the discriminator, plot generated images, save generator model

In [ ]:
def summarize_performance(epoch, g_model, d_model, dataset, latent_dim, n_samples=100):

---
## Step 46 — prepare real samples

In [ ]:
X_real, y_real = generate_real_samples(dataset, n_samples)

---
## Step 47 — evaluate discriminator on real examples

In [ ]:
_, acc_real = d_model.evaluate(X_real, y_real, verbose=0)

---
## Step 48 — prepare fake examples

In [ ]:
x_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_samples)

---
## Step 49 — evaluate discriminator on fake examples

In [ ]:
_, acc_fake = d_model.evaluate(x_fake, y_fake, verbose=0)

---
## Step 50 — summarize discriminator performance

In [ ]:
print('>Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100, acc_fake*100))

---
## Step 51 — save plot

In [ ]:
save_plot(x_fake, epoch)

---
## Step 52 — save the generator model tile file

In [ ]:
filename = 'generator_model_%03d.h5' % (epoch+1)
	g_model.save(filename)

---
## Step 53 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset.shape[0] / n_batch)
	half_batch = int(n_batch / 2)

---
## Step 54 — manually enumerate epochs

In [ ]:
for i in range(n_epochs):

---
## Step 55 — enumerate batches over the training set

In [ ]:
for j in range(bat_per_epo):

---
## Step 56 — get randomly selected 'real' samples

In [ ]:
X_real, y_real = generate_real_samples(dataset, half_batch)

---
## Step 57 — update discriminator model weights

In [ ]:
d_loss1, _ = d_model.train_on_batch(X_real, y_real)

---
## Step 58 — generate 'fake' examples

In [ ]:
X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 59 — update discriminator model weights

In [ ]:
d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)

---
## Step 60 — prepare points in latent space as input for the generator

In [ ]:
X_gan = generate_latent_points(latent_dim, n_batch)

---
## Step 61 — create inverted labels for the fake samples

In [ ]:
y_gan = ones((n_batch, 1))

---
## Step 62 — update the generator via the discriminator's error

In [ ]:
g_loss = gan_model.train_on_batch(X_gan, y_gan)

---
## Step 63 — summarize loss on this batch

In [ ]:
print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))

---
## Step 64 — evaluate the model performance, sometimes

In [ ]:
if (i+1) % 10 == 0:
			summarize_performance(i, g_model, d_model, dataset, latent_dim)

---
## Step 65 — size of the latent space

In [ ]:
latent_dim = 100

---
## Step 66 — create the discriminator

In [ ]:
d_model = define_discriminator()

---
## Step 67 — create the generator

In [ ]:
g_model = define_generator(latent_dim)

---
## Step 68 — create the gan

In [ ]:
gan_model = define_gan(g_model, d_model)

---
## Step 69 — load image data

In [ ]:
dataset = load_real_samples()

---
## Step 70 — train model

In [ ]:
train(g_model, d_model, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of a gan for generating faces 是机器学习中的常用技术。  
  *example of a gan for generating faces is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Gan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of a gan for generating faces
from numpy import load
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from matplotlib import pyplot

# define the standalone discriminator model
def define_discriminator(in_shape=(80,80,3)):
	model = Sequential()
	# normal
	model.add(Conv2D(128, (5,5), padding='same', input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 40x40
	model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 20x30
	model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 10x10
	model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 5x5
	model.add(Conv2D(128, (5,5), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# classifier
	model.add(Flatten())
	model.add(Dropout(0.4))
	model.add(Dense(1, activation='sigmoid'))
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

# define the standalone generator model
def define_generator(latent_dim):
	model = Sequential()
	# foundation for 5x5 feature maps
	n_nodes = 128 * 5 * 5
	model.add(Dense(n_nodes, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((5, 5, 128)))
	# upsample to 10x10
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 20x20
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 40x40
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 80x80
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# output layer 80x80x3
	model.add(Conv2D(3, (5,5), activation='tanh', padding='same'))
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	d_model.trainable = False
	# connect them
	model = Sequential()
	# add generator
	model.add(g_model)
	# add the discriminator
	model.add(d_model)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load and prepare training images
def load_real_samples():
	# load the face dataset
	data = load('img_align_celeba.npz')
	X = data['arr_0']
	# convert from unsigned ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return X

# select real samples
def generate_real_samples(dataset, n_samples):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# retrieve selected images
	X = dataset[ix]
	# generate 'real' class labels (1)
	y = ones((n_samples, 1))
	return X, y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(g_model, latent_dim, n_samples):
	# generate points in latent space
	x_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	X = g_model.predict(x_input)
	# create 'fake' class labels (0)
	y = zeros((n_samples, 1))
	return X, y

# create and save a plot of generated images
def save_plot(examples, epoch, n=10):
	# scale from [-1,1] to [0,1]
	examples = (examples + 1) / 2.0
	# plot images
	for i in range(n * n):
		# define subplot
		pyplot.subplot(n, n, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(examples[i])
	# save plot to file
	filename = 'generated_plot_e%03d.png' % (epoch+1)
	pyplot.savefig(filename)
	pyplot.close()

# evaluate the discriminator, plot generated images, save generator model
def summarize_performance(epoch, g_model, d_model, dataset, latent_dim, n_samples=100):
	# prepare real samples
	X_real, y_real = generate_real_samples(dataset, n_samples)
	# evaluate discriminator on real examples
	_, acc_real = d_model.evaluate(X_real, y_real, verbose=0)
	# prepare fake examples
	x_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_samples)
	# evaluate discriminator on fake examples
	_, acc_fake = d_model.evaluate(x_fake, y_fake, verbose=0)
	# summarize discriminator performance
	print('>Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100, acc_fake*100))
	# save plot
	save_plot(x_fake, epoch)
	# save the generator model tile file
	filename = 'generator_model_%03d.h5' % (epoch+1)
	g_model.save(filename)

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset.shape[0] / n_batch)
	half_batch = int(n_batch / 2)
	# manually enumerate epochs
	for i in range(n_epochs):
		# enumerate batches over the training set
		for j in range(bat_per_epo):
			# get randomly selected 'real' samples
			X_real, y_real = generate_real_samples(dataset, half_batch)
			# update discriminator model weights
			d_loss1, _ = d_model.train_on_batch(X_real, y_real)
			# generate 'fake' examples
			X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
			# update discriminator model weights
			d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)
			# prepare points in latent space as input for the generator
			X_gan = generate_latent_points(latent_dim, n_batch)
			# create inverted labels for the fake samples
			y_gan = ones((n_batch, 1))
			# update the generator via the discriminator's error
			g_loss = gan_model.train_on_batch(X_gan, y_gan)
			# summarize loss on this batch
			print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))
		# evaluate the model performance, sometimes
		if (i+1) % 10 == 0:
			summarize_performance(i, g_model, d_model, dataset, latent_dim)

# size of the latent space
latent_dim = 100
# create the discriminator
d_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(g_model, d_model, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 6 of 11